In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsmartdata202611")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/movie.csv"

In [0]:
df_movie = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
movie_schema = StructType(fields=[StructField("movieId", IntegerType(), False),
                                     StructField("title", StringType(), True),
                                     StructField("budget", DoubleType(), True),
                                     StructField("homePage", StringType(), True),
                                     StructField("popularity", DecimalType(), True),
                                     StructField("yearReleaseDate", IntegerType(), True),
                                     StructField("releaseDate", DateType(), True),
                                     StructField("revenue", DoubleType(), True),
                                     StructField("durationTime", IntegerType(), True),
                                     StructField("movieStatus", StringType(), True),
                                     StructField("voteAverage", IntegerType(), True),
                                     StructField("voteCount", IntegerType(), True)
])

In [0]:
df_movie_final = spark.read.option('header', True).schema(movie_schema).csv(ruta)

In [0]:
movie_selected_df = df_movie_final.select(col("movieId"), 
                                                col("title"), 
                                                col("budget"), 
                                                col("homePage"),
                                                col("popularity"),
                                                col("yearReleaseDate"), 
                                                col("releaseDate"), 
                                                col("revenue"), 
                                                col("durationTime"), 
                                                col("movieStatus"),
                                                col("voteAverage"),
                                                col("voteCount") )

In [0]:
movie_renamed_df = movie_selected_df.withColumnRenamed("movieId", "movie_Id") \
                                            .withColumnRenamed("title", "title") \
                                            .withColumnRenamed("budget", "budget") \
                                            .withColumnRenamed("releaseDate", "release_Date") \
                                            .withColumnRenamed("revenue", "revenue") 

In [0]:
movie_final_df = movie_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
movie_final_df.write.mode("overwrite").option("mergeSchema", "true").insertInto(f"{catalogo}.{esquema}.movie")